# DEWA Premise ID Audit

This notebook can audit DEWA premise IDs from three sources:

1. The most recent Ejari creation run's successful `progress.json` records.
2. One or more specified/uploaded Ejari creation `progress.json` success files. Multiple valid files are merged into one temporary progress file before audit.
3. Live API audit sources selected by checkbox: active contract history, non-pending contract history, and/or owner-assets leased/rented lists for the configured Emirates IDs.

For the success-file modes, provide creation `progress.json` files with top-level Emirates IDs and `ejari_creation_success` entries. Single per-property `success_*.json` detail files are not valid inputs for this mode.

Outputs are written to `runs/dewa_premise_audit_<timestamp>/`, `runs/dewa_premise_audit_successes_<timestamp>/`, or `runs/dewa_uploaded_success_progress_merge_<timestamp>/`.


## Configuration

Loads `.env`/Colab userdata, configures the five known Emirates IDs, and defines request timeout settings.


In [ ]:
# @title Configuration and environment
import base64
import csv
import json
import os
import time
from datetime import datetime
from pathlib import Path

import requests

from notebook_config import (
    DDA_OWNER_EMIRATES_IDS,
    OWNER_EMIRATES_IDS,
    PERSONAL_OWNER_EMIRATES_ID,
)
from notebook_operator_utils import (
    ask_yes_no,
    choose_multiple_options,
    choose_option,
    select_emirates_ids_for_section,
    select_file_paths,
)

try:
    import google.colab.userdata as userdata
except ImportError:
    userdata = None


PROPERTY_LIST_TYPES = ("leased", "rented")
PROPERTY_TYPE_IDS = (2, 3)
REQUEST_TIMEOUT_SECONDS = 90
DEWA_STATUS_CHECK_TIMEOUT_SECONDS = 20
CURRENT_BEARER_TOKEN = None


def load_local_env(path=".env"):
    env_path = Path(path)
    if not env_path.exists():
        return
    with env_path.open("r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_local_env()


def get_secret(name, default=None):
    if userdata is not None:
        value = userdata.get(name)
        if value is not None:
            return value
    return os.environ.get(name, default)


def require_secret(name):
    value = get_secret(name)
    if not value:
        raise RuntimeError(f"Missing required secret/env var: {name}")
    return value


BASIC_AUTH = require_secret("BASIC_AUTH")
CONSUMER_ID = require_secret("CONSUMER_ID")
DEWA_BASE_URL = require_secret("DEWA_BASE_URL").rstrip("/")
DLD_BASE_URL = require_secret("DLD_BASE_URL").rstrip("/")
DLD_PROXY_URL = require_secret("DLD_PROXY_URL").rstrip("/")
IDS_BASE_URL = require_secret("IDS_BASE_URL")

print("Configured owners:")
for eid in OWNER_EMIRATES_IDS:
    print("-", eid)


## Authentication And API Safety

Defines API error handling, bearer-token refresh behavior, and response validation helpers.


In [ ]:
# @title Authentication and API safety helpers
class ApiRequestError(Exception):
    def __init__(self, message, *, url=None, status_code=None, response=None):
        super().__init__(message)
        self.url = url
        self.status_code = status_code
        self.response = response


def response_payload(response):
    try:
        return response.json()
    except Exception:
        return response.text


def compact_payload(value, max_chars=1500):
    if isinstance(value, str):
        text = value
    else:
        text = json.dumps(value, ensure_ascii=False, default=str)
    return text if len(text) <= max_chars else text[:max_chars] + "..."


def get_bearer_token():
    url = IDS_BASE_URL
    headers = {
        "Authorization": f"Basic {BASIC_AUTH}",
        "Content-Type": "application/x-www-form-urlencoded",
    }
    response = requests.post(url, headers=headers, data={}, timeout=REQUEST_TIMEOUT_SECONDS)
    data = response_payload(response)
    if response.status_code >= 400:
        raise ApiRequestError(
            f"iPaaS bearer token failed HTTP {response.status_code}: {compact_payload(data)}",
            url=url,
            status_code=response.status_code,
            response=data,
        )
    token = data.get("id_token") if isinstance(data, dict) else None
    if not token:
        raise ApiRequestError(f"iPaaS bearer token response has no id_token: {compact_payload(data)}", url=url, response=data)

    global CURRENT_BEARER_TOKEN
    CURRENT_BEARER_TOKEN = token
    return token


def get_active_bearer_token(force_refresh=False):
    global CURRENT_BEARER_TOKEN
    if force_refresh or not CURRENT_BEARER_TOKEN:
        return get_bearer_token()
    return CURRENT_BEARER_TOKEN


def request_with_bearer(method, url, headers=None, retry_on_401=True, **kwargs):
    request_headers = dict(headers or {})
    request_headers["Authorization"] = f"Bearer {get_active_bearer_token()}"
    kwargs.setdefault("timeout", REQUEST_TIMEOUT_SECONDS)
    response = requests.request(method, url, headers=request_headers, **kwargs)

    if response.status_code == 401 and retry_on_401:
        print("iPaaS bearer token expired; refreshing and retrying request once.")
        request_headers["Authorization"] = f"Bearer {get_active_bearer_token(force_refresh=True)}"
        response = requests.request(method, url, headers=request_headers, **kwargs)

    return response


def api_errors(payload):
    if not isinstance(payload, dict):
        return []
    errors = []
    response_code = payload.get("ResponseCode") or payload.get("responseCode")
    if response_code is not None:
        normalized_response_code = str(response_code).strip().lower()
        if normalized_response_code not in {"success", "200", "ok"}:
            errors.append(f"ResponseCode={response_code}")
    if payload.get("ErrorMessage"):
        errors.append(str(payload.get("ErrorMessage")))
    for error in payload.get("ValidationErrorsList") or payload.get("validationErrorsList") or []:
        if not isinstance(error, dict):
            errors.append(str(error))
            continue
        error_number = error.get("ErrorNumber") or error.get("errorNumber")
        message_set = error.get("ErrorMessageSet") or error.get("errorMessageSet") or {}
        message = error.get("ErrorMessage") or error.get("errorMessage")
        if isinstance(message_set, dict):
            message = message or message_set.get("EnglishName") or message_set.get("englishName") or message_set.get("ArabicName")
        if error_number not in (None, 0, "0"):
            errors.append(f"ErrorNumber={error_number}; {message or compact_payload(error)}")
        elif message and str(message).strip().lower() not in {"no errors found", "no errors found."}:
            errors.append(str(message))
    return errors


def assert_ok_response(response, api_name, url):
    data = response_payload(response)
    if response.status_code >= 400:
        raise ApiRequestError(
            f"{api_name} failed HTTP {response.status_code}: {compact_payload(data)}",
            url=url,
            status_code=response.status_code,
            response=data,
        )
    errors = api_errors(data)
    if errors:
        raise ApiRequestError(
            f"{api_name} returned API errors: {' | '.join(errors)}",
            url=url,
            status_code=response.status_code,
            response=data,
        )
    return data


def get_dld_token(emirates_id):
    bearer_token = get_active_bearer_token()
    url = f"{DLD_BASE_URL}/generaltokenservice/1.0.0/auth"
    auth_obj = {
        "Method": "EmiratesId",
        "MethodIdentity": str(emirates_id),
        "MethodPasscode": "",
        "DeviceKey": "MyPC",
        "ApplicationKey": "DubaiNow",
        "Platform": "Mobile",
    }
    encoded = base64.b64encode(json.dumps(auth_obj).encode()).decode()
    headers = {
        "consumer-id": CONSUMER_ID,
        "x-nv-header": encoded,
        "Content-Type": "application/json",
        "Authorization": f"Bearer {bearer_token}",
    }
    response = request_with_bearer("post", url, headers=headers)
    data = assert_ok_response(response, "DLD token", url)
    token = data.get("token") if isinstance(data, dict) else None
    if not token:
        raise ApiRequestError(f"DLD token response has no token: {compact_payload(data)}", url=url, response=data)
    return token


## Extraction Helpers

Normalizes names, IDs, row values, DEWA premise values, and contract status fields across inconsistent API payloads.


In [ ]:
# @title Data extraction helpers
def display_name(value):
    if isinstance(value, dict):
        return value.get("EnglishName") or value.get("englishName") or value.get("ArabicName") or value.get("arabicName")
    return value


def join_values(values):
    cleaned = []
    for value in values or []:
        if value is None:
            continue
        text = str(value).strip()
        if text and text not in cleaned:
            cleaned.append(text)
    return "; ".join(cleaned)


def first_present(*values):
    for value in values:
        if value is None:
            continue
        if isinstance(value, str) and not value.strip():
            continue
        return value
    return None


def nested_get(value, *keys):
    current = value
    for key in keys:
        if isinstance(current, dict):
            current = current.get(key)
        elif isinstance(current, list) and isinstance(key, int) and len(current) > key:
            current = current[key]
        else:
            return None
    return current


def extract_dewa_premises(value):
    premises = []

    def add(candidate):
        if candidate is None:
            return
        if isinstance(candidate, (dict, list)):
            return
        text = str(candidate).strip()
        if text and text not in premises:
            premises.append(text)

    def visit(item, path=()):
        if isinstance(item, dict):
            for key, inner in item.items():
                lower_key = str(key).lower()
                lower_path = "/".join(str(part).lower() for part in path + (key,))
                if lower_key in {"premisenumber", "premiseno", "premise_no", "premiseid", "premise_id"}:
                    add(inner)
                if lower_key in {"accountnumber", "account_number"} and "dewa" in lower_path:
                    add(inner)
                visit(inner, path + (key,))
        elif isinstance(item, list):
            for index, inner in enumerate(item):
                visit(inner, path + (str(index),))

    visit(value)
    return premises



def first_dewa_premise(value):
    premises = extract_dewa_premises(value)
    return premises[0] if premises else None


def split_premise_values(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        raw_values = value
    else:
        raw_values = str(value).replace("\n", ";").split(";")

    values = []
    for item in raw_values:
        text = str(item or "").strip()
        if text and text not in values:
            values.append(text)
    return values


def dewa_premise_mismatch_columns(source_values):
    populated_sources = {}
    unique_values = []
    for source_name, value in source_values.items():
        values = split_premise_values(value)
        if not values:
            continue
        populated_sources[source_name] = values
        for premise_value in values:
            if premise_value not in unique_values:
                unique_values.append(premise_value)

    has_mismatch = len(unique_values) > 1
    return {
        "dewa_premise_id_mismatch": "yes" if has_mismatch else "no",
        "dewa_premise_id_mismatch_detail": (
            " | ".join(f"{source}={join_values(values)}" for source, values in populated_sources.items())
            if has_mismatch
            else ""
        ),
    }


def first_exact_key_value(value, target_key):
    if isinstance(value, dict):
        for key, inner in value.items():
            if str(key) == target_key:
                if inner is None:
                    continue
                if isinstance(inner, str) and not inner.strip():
                    continue
                return inner
        for inner in value.values():
            found = first_exact_key_value(inner, target_key)
            if found is not None:
                return found
    elif isinstance(value, list):
        for inner in value:
            found = first_exact_key_value(inner, target_key)
            if found is not None:
                return found
    return None


def contract_version_value(value):
    if value is None:
        return None
    direct_version = first_present(
        nested_get(value, "VersionNumber"),
        nested_get(value, "ContractDetails", "VersionNumber"),
        nested_get(value, "Response", "VersionNumber"),
        nested_get(value, "Response", "ContractDetails", "VersionNumber"),
        nested_get(value, "response", "Response", "VersionNumber"),
        nested_get(value, "response", "Response", "ContractDetails", "VersionNumber"),
        nested_get(value, "api5_response", "Response", "VersionNumber"),
    )
    if direct_version is not None:
        return direct_version

    recursive_version = first_exact_key_value(value, "VersionNumber")
    if recursive_version is not None:
        return recursive_version

    return first_present(
        nested_get(value, "Response", "AssociatedTenancyContract", "Version"),
        nested_get(value, "response", "Response", "AssociatedTenancyContract", "Version"),
        nested_get(value, "AssociatedTenancyContract", "Version"),
    )


def numeric_contract_version(value):
    if value is None:
        return None
    try:
        return float(str(value).strip())
    except Exception:
        return None


def contract_version_report_fields(*sources):
    version = first_present(*(contract_version_value(source) for source in sources))
    numeric_version = numeric_contract_version(version)
    if numeric_version is None:
        version_type = ""
    else:
        version_type = "Renewal" if numeric_version > 1 else "New"
    return {
        "contract_version_number": version if version is not None else "",
        "contract_version_type": version_type,
    }


def property_title_from_sources(*sources):
    for source in sources:
        if not isinstance(source, dict):
            continue
        candidate = first_present(
            display_name(source.get("Title")),
            source.get("title"),
            source.get("property_title"),
            source.get("PropertyName"),
            source.get("propertyName"),
            source.get("property_name"),
            display_name(nested_get(source, "Property", "Title")),
            display_name(nested_get(source, "AssociatedProperty", "Title")),
            display_name(nested_get(source, "Response", "Title")),
            display_name(nested_get(source, "Response", "Property", "Title")),
            display_name(nested_get(source, "Response", "AssociatedProperty", "Title")),
            display_name(nested_get(source, "response", "Response", "Title")),
            display_name(nested_get(source, "response", "Response", "Property", "Title")),
            display_name(nested_get(source, "response", "Response", "AssociatedProperty", "Title")),
        )
        if candidate is not None and str(candidate).strip():
            return candidate
    return ""


def premise_status_message_text(message):
    if isinstance(message, dict):
        return join_values(message.values())
    return message


def premise_status_item(response):
    items = nested_get(response, "MT_PremiseStatusCheck_Resp", "Item")
    if isinstance(items, list) and items:
        return items[0] if isinstance(items[0], dict) else {}
    if isinstance(items, dict):
        return items
    return {}


def premise_status_report_fields(result, input_premise_no=None):
    if not isinstance(result, dict):
        return {
            "dewa_status_check_input_premise_no": input_premise_no,
            "dewa_status_check_error": "Premise status check was not called.",
        }

    item = result.get("item") if isinstance(result.get("item"), dict) else premise_status_item(result.get("response"))
    response = result.get("response") if isinstance(result.get("response"), dict) else {}
    response_message = nested_get(response, "MT_PremiseStatusCheck_Resp", "Message")

    return {
        "dewa_status_check_input_premise_no": input_premise_no,
        "dewa_status_check_http_status": result.get("status_code"),
        "dewa_status_check_result": item.get("Result"),
        "dewa_status_check_message": premise_status_message_text(item.get("Message")) or premise_status_message_text(response_message),
        "dewa_status_check_sd_status": item.get("SDStatus"),
        "dewa_status_check_bp_number": item.get("BPNumber"),
        "dewa_status_check_contract_act_no": item.get("ContractActNo"),
        "dewa_status_check_contract_act_name": item.get("ContractActName"),
        "dewa_status_check_ejari_number": item.get("EJARINumber"),
        "dewa_status_check_premise_no": item.get("PremiseNo"),
        "dewa_status_check_output1": item.get("Output1"),
        "dewa_status_check_output2": item.get("Output2"),
        "dewa_status_check_output3": item.get("Output3"),
        "dewa_status_check_output4": item.get("Output4"),
        "dewa_status_check_ad_output": item.get("AdOutput"),
        "dewa_status_check_error": result.get("error", ""),
    }

def contract_status_text(contract):
    return display_name(contract.get("ContractStatus") or contract.get("Status")) if isinstance(contract, dict) else None


def make_output_dir(prefix="dewa_premise_audit"):
    path = Path("runs") / f"{prefix}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    path.mkdir(parents=True, exist_ok=True)
    return path


## DLD And DEWA Calls

Calls owner-assets lists/details, contract history, contract details, and DEWA premise status APIs.


In [ ]:
# @title DLD and DEWA API calls
def dld_headers(dld_token, *, accept_text=False):
    headers = {
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
    }
    if accept_text:
        headers["accept"] = "text/plain"
    return headers


def get_properties_list(dld_token, list_type, property_type_id):
    url = f"{DLD_BASE_URL}/generalservices/1.0.0/owner-assets/{list_type}/{property_type_id}"
    response = request_with_bearer("get", url, headers=dld_headers(dld_token))
    data = assert_ok_response(response, f"Properties list {list_type}/{property_type_id}", url)
    properties = nested_get(data, "Response", "Properties")
    if not isinstance(properties, list):
        raise ApiRequestError(f"Properties list {list_type}/{property_type_id} returned no Response.Properties list", url=url, response=data)
    return properties, data, url


def get_property_detail(dld_token, list_type, property_type_id, property_row_value):
    # The list-specific detail endpoint works for rented/leased; owned is kept as fallback because Postman has examples for it.
    errors = []
    for detail_list_type in [list_type, "owned"]:
        url = f"{DLD_BASE_URL}/generalservices/1.0.0/owner-assets/{detail_list_type}/{property_type_id}/{property_row_value}"
        response = request_with_bearer("get", url, headers=dld_headers(dld_token))
        data = response_payload(response)
        if response.status_code < 400 and isinstance(data, dict) and isinstance(data.get("Response"), dict):
            response_errors = api_errors(data)
            if not response_errors:
                return data, url, detail_list_type
            errors.append(f"{detail_list_type}: {' | '.join(response_errors)}")
        else:
            errors.append(f"{detail_list_type}: HTTP {response.status_code}; {compact_payload(data, 400)}")
    raise ApiRequestError("Property detail failed: " + " || ".join(errors), response=errors)


def normalized_contract_status(contract):
    return str(contract_status_text(contract) or "").strip().lower()


def is_active_contract_history_record(contract):
    return normalized_contract_status(contract) == "active"


def is_pending_contract_history_record(contract):
    return normalized_contract_status(contract) == "pending"


def include_contract_history_record(contract, status_filter="active"):
    status_filter = str(status_filter or "active").strip().lower()
    if status_filter in {"all", "all-statuses", "any"}:
        return True
    if status_filter in {"active", "only-active"}:
        return is_active_contract_history_record(contract)
    if status_filter in {"non-pending", "not-pending", "exclude-pending", "all-other-than-pending"}:
        return not is_pending_contract_history_record(contract)
    raise ValueError(f"Unsupported contract history status filter: {status_filter}")


def get_contract_history(dld_token, source="actual", status_filter="active"):
    if source == "proxy":
        url = f"{DLD_PROXY_URL}/ejariservices/properties/getcontracts"
    else:
        url = f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/properties/getcontracts"
    response = request_with_bearer("get", url, headers=dld_headers(dld_token, accept_text=True))
    data = assert_ok_response(response, f"Contract history ({source})", url)
    response_data = data.get("Response") if isinstance(data, dict) else {}
    owner_contracts = response_data.get("OwnerContracts") if isinstance(response_data, dict) else []
    tenant_contracts = response_data.get("TenantContracts") if isinstance(response_data, dict) else []
    owner_contracts = owner_contracts if isinstance(owner_contracts, list) else []
    tenant_contracts = tenant_contracts if isinstance(tenant_contracts, list) else []

    contracts = []
    for role, role_contracts in (("owner", owner_contracts), ("tenant", tenant_contracts)):
        for contract in role_contracts:
            if not isinstance(contract, dict) or not include_contract_history_record(contract, status_filter=status_filter):
                continue
            item = dict(contract)
            item["_history_role"] = role
            item["_history_status"] = normalized_contract_status(contract)
            item["_history_status_filter"] = status_filter
            contracts.append(item)
    return contracts, data, url


def get_contract_details(dld_token, contract_row_value):
    url = f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/contracts/{contract_row_value}"
    headers = dld_headers(dld_token, accept_text=True)
    headers["Content-Type"] = "application/json-patch+json"
    response = request_with_bearer("get", url, headers=headers)
    data = assert_ok_response(response, "Contract details", url)
    details = nested_get(data, "Response", "ContractDetails")
    if not isinstance(details, dict):
        raise ApiRequestError("Contract details response has no Response.ContractDetails object", url=url, response=data)
    return details, data, url



def premise_status_check(contract_number, premise_no):
    url = f"{DEWA_BASE_URL}/moveinprocess/rest/1.0.0/PremiseStatus_Check"
    headers = {
        "Content-Type": "application/json",
    }
    payload = {
        "MT_PremiseStatusCheck_Req": {
            "LangKey": "EN",
            "Vendor": "DLD",
            "Item": {
                "EJARINumber": str(contract_number),
                "PremiseNo": str(premise_no),
                "Input1": "",
                "Input2": "",
                "Input3": "",
                "Input4": "",
                "AddnlInput": "",
            },
        }
    }
    response = request_with_bearer("post", url, headers=headers, json=payload, timeout=DEWA_STATUS_CHECK_TIMEOUT_SECONDS)
    data = response_payload(response)
    result = {
        "status_code": response.status_code,
        "response": data,
        "item": premise_status_item(data),
        "url": url,
        "payload": payload,
    }
    if response.status_code >= 400:
        result["error"] = f"Premise status check failed HTTP {response.status_code}: {compact_payload(data)}"
    return result


## Matching Logic

Builds indexes and joins contract-history rows with leased/rented property-list rows.


In [ ]:
# @title Matching and row building
def add_index(index, bucket, key, contract):
    if key is None:
        return
    text = str(key).strip()
    if not text:
        return
    index.setdefault(bucket, {}).setdefault(text, []).append(contract)


def build_history_index(contracts):
    index = {}
    for contract in contracts:
        add_index(index, "contract_row_value", contract.get("AssociatedContractRowValue"), contract)
        add_index(index, "contract_row_value", contract.get("DataRowValue"), contract)
        add_index(index, "property_row_value", contract.get("RowValue"), contract)
        add_index(index, "property_row_value", contract.get("PropertyRowValue"), contract)
        add_index(index, "property_id", contract.get("PropertyId"), contract)
        add_index(index, "contract_number", contract.get("ContractNumber"), contract)
    return index


def first_history_match(prop, contract_row_value, contract_number, history_index):
    lookup_order = [
        ("contract_row_value", contract_row_value),
        ("property_id", prop.get("PropertyId")),
        ("property_row_value", prop.get("RowValue")),
        ("contract_number", contract_number),
    ]
    for bucket, key in lookup_order:
        if key is None:
            continue
        matches = history_index.get(bucket, {}).get(str(key).strip()) or []
        if matches:
            return matches[0]
    return None


def get_contract_row_value_from_sources(prop, property_detail_response, history_match):
    return first_present(
        prop.get("AssociatedContractRowValue"),
        nested_get(property_detail_response, "Response", "AssociatedTenancyContract", "RowValue"),
        history_match.get("AssociatedContractRowValue") if isinstance(history_match, dict) else None,
        history_match.get("DataRowValue") if isinstance(history_match, dict) else None,
    )


def get_contract_number_from_sources(prop, property_detail_response, contract_details, history_match):
    return first_present(
        prop.get("ContractNumber"),
        nested_get(property_detail_response, "Response", "AssociatedTenancyContract", "ContractNumber"),
        contract_details.get("ContractNumber") if isinstance(contract_details, dict) else None,
        history_match.get("ContractNumber") if isinstance(history_match, dict) else None,
    )


def contract_history_property_type_id(contract):
    return first_present(
        contract.get("PropertyTypeId"),
        contract.get("PropertyTypeID"),
        nested_get(contract, "PropertyType", "Identity"),
        nested_get(contract, "PropertyType", "Value"),
    )


def fetch_target_properties(dld_token):
    all_properties = []
    counts = {}
    list_urls = {}
    for list_type in PROPERTY_LIST_TYPES:
        for property_type_id in PROPERTY_TYPE_IDS:
            properties, _, url = get_properties_list(dld_token, list_type, property_type_id)
            counts[f"{list_type}/{property_type_id}"] = len(properties)
            list_urls[f"{list_type}/{property_type_id}"] = url
            for prop in properties:
                if not isinstance(prop, dict):
                    continue
                item = dict(prop)
                item["_properties_list_type"] = list_type
                item["_property_type_id"] = property_type_id
                item["_properties_list_url"] = url
                all_properties.append(item)
    return all_properties, counts, list_urls


def get_cached_contract_details(dld_token, contract_row_value, contract_details_cache):
    if not contract_row_value:
        return None, None, ""
    if contract_row_value in contract_details_cache:
        cached = contract_details_cache[contract_row_value]
        return cached.get("details"), cached.get("url"), cached.get("error", "")
    try:
        contract_details, _, contract_details_url = get_contract_details(dld_token, contract_row_value)
        contract_details_cache[contract_row_value] = {"details": contract_details, "url": contract_details_url, "error": ""}
        return contract_details, contract_details_url, ""
    except Exception as exc:
        contract_details_error = str(exc)
        contract_details_cache[contract_row_value] = {"details": None, "url": None, "error": contract_details_error}
        return None, None, contract_details_error


def get_cached_premise_status(contract_number, premise_no, premise_status_cache):
    if not contract_number or not premise_no:
        return {
            "status_code": None,
            "response": None,
            "item": {},
            "error": "Missing contract number or DEWA premise number for PremiseStatus_Check.",
        }

    cache_key = (str(contract_number), str(premise_no))
    if cache_key not in premise_status_cache:
        try:
            premise_status_cache[cache_key] = premise_status_check(contract_number, premise_no)
        except Exception as exc:
            premise_status_cache[cache_key] = {
                "status_code": None,
                "response": None,
                "item": {},
                "error": f"PremiseStatus_Check request failed: {exc}",
            }
    return premise_status_cache[cache_key]


def emit_audit_row(row, rows=None, row_writer=None):
    if rows is not None:
        rows.append(row)
    if row_writer is not None:
        row_writer(row)


def make_presence_columns(
    present_in_properties_list,
    present_in_contract_history,
    *,
    properties_list_checked=True,
    contract_history_checked=True,
):
    if properties_list_checked and contract_history_checked:
        if present_in_properties_list and present_in_contract_history:
            source_record_type = "matched"
            source_mismatch = ""
        elif present_in_properties_list:
            source_record_type = "properties_list_only"
            source_mismatch = "PROPERTY_LIST_ONLY"
        else:
            source_record_type = "contract_history_only"
            source_mismatch = "CONTRACT_HISTORY_ONLY"
    elif properties_list_checked:
        source_record_type = "properties_list"
        source_mismatch = ""
    elif contract_history_checked:
        source_record_type = "contract_history"
        source_mismatch = ""
    else:
        source_record_type = "not_checked"
        source_mismatch = ""

    return {
        "source_record_type": source_record_type,
        "present_in_properties_list": "yes" if present_in_properties_list else ("no" if properties_list_checked else "not checked"),
        "present_in_contract_history": "yes" if present_in_contract_history else ("no" if contract_history_checked else "not checked"),
        "leased_rented_properties_presence": "present" if present_in_properties_list else ("missing from leased/rented properties list" if properties_list_checked else "not checked"),
        "contract_history_presence": "present" if present_in_contract_history else ("missing from contract history" if contract_history_checked else "not checked"),
        "source_mismatch": source_mismatch,
    }


LIVE_DEWA_SOURCE_CONTRACT_HISTORY_ACTIVE = "contract-history-active"
LIVE_DEWA_SOURCE_CONTRACT_HISTORY_NON_PENDING = "contract-history-non-pending"
LIVE_DEWA_SOURCE_OWNER_ASSETS = "owner-assets-leased-rented"
DEFAULT_LIVE_DEWA_AUDIT_SOURCES = (
    LIVE_DEWA_SOURCE_CONTRACT_HISTORY_ACTIVE,
    LIVE_DEWA_SOURCE_OWNER_ASSETS,
)
LIVE_DEWA_AUDIT_SOURCE_LABELS = {
    LIVE_DEWA_SOURCE_CONTRACT_HISTORY_ACTIVE: "contract history (only active)",
    LIVE_DEWA_SOURCE_CONTRACT_HISTORY_NON_PENDING: "contract history (all other than pending)",
    LIVE_DEWA_SOURCE_OWNER_ASSETS: "owner assets (leased + rented)",
}


def normalize_live_dewa_audit_sources(live_audit_sources=None):
    source_values = set(LIVE_DEWA_AUDIT_SOURCE_LABELS)
    aliases = {
        "active": LIVE_DEWA_SOURCE_CONTRACT_HISTORY_ACTIVE,
        "contract-history-active": LIVE_DEWA_SOURCE_CONTRACT_HISTORY_ACTIVE,
        "non-pending": LIVE_DEWA_SOURCE_CONTRACT_HISTORY_NON_PENDING,
        "not-pending": LIVE_DEWA_SOURCE_CONTRACT_HISTORY_NON_PENDING,
        "contract-history-non-pending": LIVE_DEWA_SOURCE_CONTRACT_HISTORY_NON_PENDING,
        "owner-assets": LIVE_DEWA_SOURCE_OWNER_ASSETS,
        "leased-rented": LIVE_DEWA_SOURCE_OWNER_ASSETS,
        "owner-assets-leased-rented": LIVE_DEWA_SOURCE_OWNER_ASSETS,
    }
    if live_audit_sources is None:
        live_audit_sources = DEFAULT_LIVE_DEWA_AUDIT_SOURCES
    if isinstance(live_audit_sources, str):
        raw_sources = [value.strip() for value in live_audit_sources.replace(";", ",").split(",")]
    else:
        raw_sources = list(live_audit_sources)

    selected = []
    seen = set()
    for source in raw_sources:
        source_value = aliases.get(str(source).strip().lower(), str(source).strip())
        if source_value not in source_values:
            raise ValueError(f"Unsupported live DEWA audit source: {source}")
        if source_value not in seen:
            seen.add(source_value)
            selected.append(source_value)
    if not selected:
        raise ValueError("Select at least one live DEWA audit source.")
    return tuple(selected)


def contract_history_identity_key(contract):
    contract_row_value = first_present(contract.get("AssociatedContractRowValue"), contract.get("DataRowValue"))
    contract_number = contract.get("ContractNumber")
    property_id = contract.get("PropertyId")
    property_row_value = first_present(contract.get("RowValue"), contract.get("PropertyRowValue"))
    role = contract.get("_history_role")
    if contract_row_value:
        return role, "contract_row_value", str(contract_row_value)
    if contract_number:
        return role, "contract_number", str(contract_number)
    return role, "property", str(property_id), str(property_row_value), normalized_contract_status(contract)


def selected_contract_history_scopes(contract, live_audit_sources):
    scopes = []
    if LIVE_DEWA_SOURCE_CONTRACT_HISTORY_ACTIVE in live_audit_sources and is_active_contract_history_record(contract):
        scopes.append("active")
    if LIVE_DEWA_SOURCE_CONTRACT_HISTORY_NON_PENDING in live_audit_sources and not is_pending_contract_history_record(contract):
        scopes.append("non_pending_excluding_pending")
    return scopes


def filter_selected_contract_history(raw_contracts, live_audit_sources):
    merged = {}
    for contract in raw_contracts:
        scopes = selected_contract_history_scopes(contract, live_audit_sources)
        if not scopes:
            continue
        key = contract_history_identity_key(contract)
        if key not in merged:
            item = dict(contract)
            item["_history_source_scopes"] = []
            merged[key] = item
        for scope in scopes:
            if scope not in merged[key]["_history_source_scopes"]:
                merged[key]["_history_source_scopes"].append(scope)
        merged[key]["_history_status_scope"] = ",".join(merged[key]["_history_source_scopes"])
    return list(merged.values())


def audit_owner_dewa_premises(emirates_id, history_source="actual", live_audit_sources=None, row_writer=None):
    live_audit_sources = normalize_live_dewa_audit_sources(live_audit_sources)
    use_owner_assets = LIVE_DEWA_SOURCE_OWNER_ASSETS in live_audit_sources
    use_contract_history = any(
        source in live_audit_sources
        for source in (LIVE_DEWA_SOURCE_CONTRACT_HISTORY_ACTIVE, LIVE_DEWA_SOURCE_CONTRACT_HISTORY_NON_PENDING)
    )
    live_audit_source_text = ", ".join(LIVE_DEWA_AUDIT_SOURCE_LABELS[source] for source in live_audit_sources)

    print(f"\nProcessing owner Emirates ID: {emirates_id}")
    print(f"Live audit sources: {live_audit_source_text}")
    get_active_bearer_token(force_refresh=True)
    dld_token = get_dld_token(emirates_id)

    raw_history_contracts = []
    history_contracts = []
    history_url = ""
    if use_contract_history:
        raw_history_contracts, _, history_url = get_contract_history(dld_token, source=history_source, status_filter="all")
        history_contracts = filter_selected_contract_history(raw_history_contracts, live_audit_sources)
        print(f"Contract history records fetched: {len(raw_history_contracts)} | selected: {len(history_contracts)}")
    else:
        print("Contract history source not selected.")
    history_index = build_history_index(history_contracts)

    properties = []
    counts = {}
    if use_owner_assets:
        properties, counts, _ = fetch_target_properties(dld_token)
        total_property_list_count = sum(counts.values())
        print("Property list counts:", counts)
        print(f"Total leased+rented property rows: {total_property_list_count}")
    else:
        total_property_list_count = 0
        print("Owner-assets leased+rented source not selected.")

    rows = [] if row_writer is None else None
    output_rows = 0
    contract_details_cache = {}
    premise_status_cache = {}
    seen_property_keys = set()
    matched_history_ids = set()
    matched_property_rows = 0
    property_list_only_rows = 0

    for prop in properties:
        list_type = prop.get("_properties_list_type")
        property_type_id = prop.get("_property_type_id")
        property_id = prop.get("PropertyId")
        property_row_value = prop.get("RowValue")
        initial_contract_row_value = prop.get("AssociatedContractRowValue")
        property_key = (str(property_id), str(property_row_value), str(initial_contract_row_value), str(list_type), str(property_type_id))
        if property_key in seen_property_keys:
            continue
        seen_property_keys.add(property_key)

        title = property_title_from_sources(prop)
        property_detail_response = None
        property_detail_url = None
        property_detail_list_type_used = None
        property_detail_error = ""

        history_match = first_history_match(prop, initial_contract_row_value, prop.get("ContractNumber"), history_index)

        try:
            if property_row_value:
                property_detail_response, property_detail_url, property_detail_list_type_used = get_property_detail(
                    dld_token,
                    list_type,
                    property_type_id,
                    property_row_value,
                )
        except Exception as exc:
            property_detail_error = str(exc)

        contract_row_value = get_contract_row_value_from_sources(prop, property_detail_response, history_match)
        if not history_match:
            history_match = first_history_match(prop, contract_row_value, None, history_index)

        contract_details, contract_details_url, contract_details_error = get_cached_contract_details(
            dld_token,
            contract_row_value,
            contract_details_cache,
        )

        contract_number = get_contract_number_from_sources(prop, property_detail_response, contract_details, history_match)
        if history_match is None:
            history_match = first_history_match(prop, contract_row_value, contract_number, history_index)

        title = property_title_from_sources(prop, property_detail_response, contract_details, history_match)
        contract_version_fields = contract_version_report_fields(contract_details, history_match, prop, property_detail_response)

        if history_match is not None:
            matched_history_ids.add(id(history_match))
            matched_property_rows += 1
        else:
            property_list_only_rows += 1

        dewa_premise_from_property_detail = join_values(extract_dewa_premises(property_detail_response))
        dewa_premise_from_contract_details = join_values(extract_dewa_premises(contract_details))
        dewa_premise_from_contract_history = join_values(extract_dewa_premises(history_match))
        dewa_premise_from_properties_list = join_values(extract_dewa_premises(prop))
        premise_for_status_check = first_present(
            first_dewa_premise(property_detail_response),
            first_dewa_premise(contract_details),
            first_dewa_premise(prop),
            first_dewa_premise(history_match),
        )
        premise_status_result = get_cached_premise_status(contract_number, premise_for_status_check, premise_status_cache)
        premise_status_fields = premise_status_report_fields(premise_status_result, premise_for_status_check)
        premise_mismatch = dewa_premise_mismatch_columns({
            "property_detail": dewa_premise_from_property_detail,
            "contract_details": dewa_premise_from_contract_details,
            "contract_history": dewa_premise_from_contract_history,
            "properties_list": dewa_premise_from_properties_list,
            "dewa_status_check": premise_status_fields.get("dewa_status_check_premise_no"),
        })

        row = {
            "emirates_id": emirates_id,
            "property_id": property_id,
            "property_row_value": property_row_value,
            "contract_number": contract_number,
            **contract_version_fields,
            **make_presence_columns(
                True,
                history_match is not None,
                properties_list_checked=use_owner_assets,
                contract_history_checked=use_contract_history,
            ),
            "live_audit_sources": live_audit_source_text,
            "contract_history_status_scope": history_match.get("_history_status_scope") if isinstance(history_match, dict) else "",
            "dewa_premise_from_property_detail": dewa_premise_from_property_detail,
            "dewa_premise_from_contract_details": dewa_premise_from_contract_details,
            "dewa_premise_from_contract_history": dewa_premise_from_contract_history,
            "dewa_premise_from_properties_list": dewa_premise_from_properties_list,
            **premise_mismatch,
            **premise_status_fields,
            "properties_list_type": list_type,
            "property_type_id": property_type_id,
            "property_title": title,
            "contract_row_value": contract_row_value,
            "contract_status_from_properties_list": contract_status_text(prop),
            "contract_status_from_history": contract_status_text(history_match) if isinstance(history_match, dict) else None,
            "contract_history_role": history_match.get("_history_role") if isinstance(history_match, dict) else None,
            "property_detail_list_type_used": property_detail_list_type_used,
            "properties_list_url": prop.get("_properties_list_url"),
            "property_detail_url": property_detail_url,
            "contract_details_url": contract_details_url,
            "contract_history_url": history_url,
            "property_detail_error": property_detail_error,
            "contract_details_error": contract_details_error,
        }
        emit_audit_row(row, rows, row_writer)
        output_rows += 1

    contract_history_only_rows = 0
    for history_contract in history_contracts:
        if id(history_contract) in matched_history_ids:
            continue

        contract_row_value = first_present(history_contract.get("AssociatedContractRowValue"), history_contract.get("DataRowValue"))
        contract_number = history_contract.get("ContractNumber")
        property_id = history_contract.get("PropertyId")
        property_row_value = first_present(history_contract.get("RowValue"), history_contract.get("PropertyRowValue"))
        contract_details, contract_details_url, contract_details_error = get_cached_contract_details(
            dld_token,
            contract_row_value,
            contract_details_cache,
        )
        contract_history_only_rows += 1

        resolved_contract_number = first_present(contract_number, contract_details.get("ContractNumber") if isinstance(contract_details, dict) else None)
        title = property_title_from_sources(history_contract, contract_details)
        contract_version_fields = contract_version_report_fields(contract_details, history_contract)
        dewa_premise_from_contract_details = join_values(extract_dewa_premises(contract_details))
        dewa_premise_from_contract_history = join_values(extract_dewa_premises(history_contract))
        premise_for_status_check = first_present(
            first_dewa_premise(contract_details),
            first_dewa_premise(history_contract),
        )
        premise_status_result = get_cached_premise_status(resolved_contract_number, premise_for_status_check, premise_status_cache)
        premise_status_fields = premise_status_report_fields(premise_status_result, premise_for_status_check)
        premise_mismatch = dewa_premise_mismatch_columns({
            "contract_details": dewa_premise_from_contract_details,
            "contract_history": dewa_premise_from_contract_history,
            "dewa_status_check": premise_status_fields.get("dewa_status_check_premise_no"),
        })

        row = {
            "emirates_id": emirates_id,
            "property_id": property_id,
            "property_row_value": property_row_value,
            "contract_number": resolved_contract_number,
            **contract_version_fields,
            **make_presence_columns(
                False,
                True,
                properties_list_checked=use_owner_assets,
                contract_history_checked=use_contract_history,
            ),
            "live_audit_sources": live_audit_source_text,
            "contract_history_status_scope": history_contract.get("_history_status_scope"),
            "dewa_premise_from_property_detail": "",
            "dewa_premise_from_contract_details": dewa_premise_from_contract_details,
            "dewa_premise_from_contract_history": dewa_premise_from_contract_history,
            "dewa_premise_from_properties_list": "",
            **premise_mismatch,
            **premise_status_fields,
            "properties_list_type": "",
            "property_type_id": contract_history_property_type_id(history_contract),
            "property_title": title,
            "contract_row_value": contract_row_value,
            "contract_status_from_properties_list": "",
            "contract_status_from_history": contract_status_text(history_contract),
            "contract_history_role": history_contract.get("_history_role"),
            "property_detail_list_type_used": "",
            "properties_list_url": "",
            "property_detail_url": "",
            "contract_details_url": contract_details_url,
            "contract_history_url": history_url,
            "property_detail_error": "Not fetched because this contract was not present in leased/rented properties list." if use_owner_assets else "Not fetched because owner-assets leased/rented source was not selected.",
            "contract_details_error": contract_details_error,
        }
        emit_audit_row(row, rows, row_writer)
        output_rows += 1

    print(
        f"Matched property rows: {matched_property_rows} | "
        f"property-list-only rows: {property_list_only_rows} | "
        f"contract-history-only rows: {contract_history_only_rows}"
    )

    summary = {
        "emirates_id": emirates_id,
        "history_source": history_source,
        "live_audit_sources": list(live_audit_sources),
        "contract_history_raw_count": len(raw_history_contracts),
        "contract_history_count": len(history_contracts),
        "property_counts": counts,
        "leased_rented_property_count": total_property_list_count,
        "matched_property_rows": matched_property_rows,
        "property_rows_missing_from_contract_history": property_list_only_rows if use_contract_history else 0,
        "contract_history_rows_missing_from_properties": contract_history_only_rows if use_owner_assets else 0,
        "contract_history_rows_without_owner_assets_check": contract_history_only_rows if not use_owner_assets else 0,
        "output_rows": output_rows,
    }
    return rows or [], summary


## Output Schema

Defines CSV/JSON output fields, writes incremental run files, and summarizes counts.


In [ ]:
# @title Output schema and writer
OUTPUT_FIELDS = [
    "emirates_id",
    "property_id",
    "property_row_value",
    "property_title",
    "contract_number",
    "contract_version_number",
    "contract_version_type",
    "dewa_premise_from_property_detail",
    "dewa_premise_from_contract_details",
    "dewa_premise_from_contract_history",
    "dewa_premise_from_properties_list",
    "dewa_premise_from_success_progress",
    "dewa_premise_id_mismatch",
    "dewa_premise_id_mismatch_detail",
    "dewa_status_check_input_premise_no",
    "dewa_status_check_http_status",
    "dewa_status_check_result",
    "dewa_status_check_message",
    "dewa_status_check_sd_status",
    "dewa_status_check_bp_number",
    "dewa_status_check_contract_act_no",
    "dewa_status_check_contract_act_name",
    "dewa_status_check_ejari_number",
    "dewa_status_check_premise_no",
    "dewa_status_check_output1",
    "dewa_status_check_output2",
    "dewa_status_check_output3",
    "dewa_status_check_output4",
    "dewa_status_check_ad_output",
    "dewa_status_check_error",
    "source_record_type",
    "live_audit_sources",
    "contract_history_status_scope",
    "present_in_properties_list",
    "present_in_contract_history",
    "leased_rented_properties_presence",
    "contract_history_presence",
    "source_mismatch",
    "properties_list_type",
    "property_type_id",
    "contract_row_value",
    "contract_status_from_properties_list",
    "contract_status_from_history",
    "contract_history_role",
    "property_detail_list_type_used",
    "property_detail_error",
    "contract_details_error",
    "properties_list_url",
    "property_detail_url",
    "contract_details_url",
    "contract_history_url",
    "progress_success_file",
    "progress_success_property_key",
]


def initialize_audit_outputs(output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    paths = {
        "csv_path": output_dir / "dewa_premise_audit.csv",
        "json_path": output_dir / "dewa_premise_audit.json",
        "jsonl_path": output_dir / "dewa_premise_audit.jsonl",
        "summary_path": output_dir / "dewa_premise_audit_summary.json",
    }

    with paths["csv_path"].open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS, extrasaction="ignore")
        writer.writeheader()

    paths["jsonl_path"].write_text("", encoding="utf-8")
    paths["json_path"].write_text("[]", encoding="utf-8")
    save_audit_summaries([], paths["summary_path"])
    return paths


def append_audit_row(row, csv_path, jsonl_path):
    with Path(csv_path).open("a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS, extrasaction="ignore")
        writer.writerow(row)
        f.flush()

    with Path(jsonl_path).open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")
        f.flush()


def save_audit_summaries(summaries, summary_path):
    with Path(summary_path).open("w", encoding="utf-8") as f:
        json.dump(summaries, f, ensure_ascii=False, indent=2, default=str)


def finalize_audit_json(jsonl_path, json_path):
    row_count = 0
    jsonl_path = Path(jsonl_path)
    json_path = Path(json_path)
    with json_path.open("w", encoding="utf-8") as out:
        out.write("[\n")
        first = True
        if jsonl_path.exists():
            with jsonl_path.open("r", encoding="utf-8") as src:
                for line in src:
                    line = line.strip()
                    if not line:
                        continue
                    row = json.loads(line)
                    if not first:
                        out.write(",\n")
                    out.write(json.dumps(row, ensure_ascii=False, indent=2, default=str))
                    first = False
                    row_count += 1
        out.write("\n]")
    return row_count


def save_audit_outputs(rows, summaries, output_dir):
    paths = initialize_audit_outputs(output_dir)
    for row in rows:
        append_audit_row(row, paths["csv_path"], paths["jsonl_path"])
    finalize_audit_json(paths["jsonl_path"], paths["json_path"])
    save_audit_summaries(summaries, paths["summary_path"])
    return paths["csv_path"], paths["json_path"], paths["summary_path"]


def run_dewa_premise_audit(owner_emirates_ids=OWNER_EMIRATES_IDS, history_source="actual", live_audit_sources=None):
    live_audit_sources = normalize_live_dewa_audit_sources(live_audit_sources)
    output_dir = make_output_dir()
    paths = initialize_audit_outputs(output_dir)
    summaries = []
    total_rows = 0

    print(f"Run folder: {output_dir}")
    print(f"Live CSV: {paths['csv_path']}")
    print(f"Live JSONL: {paths['jsonl_path']}")
    print("Selected live audit sources:")
    for source in live_audit_sources:
        print(f"- {LIVE_DEWA_AUDIT_SOURCE_LABELS[source]}")

    def row_writer(row):
        nonlocal total_rows
        append_audit_row(row, paths["csv_path"], paths["jsonl_path"])
        total_rows += 1

    selected_emirates_ids = select_emirates_ids_for_section("DEWA premise live audit", owner_emirates_ids, default_all=True)
    for emirates_id in selected_emirates_ids:
        started = datetime.now().isoformat(timespec="seconds")
        try:
            _, summary = audit_owner_dewa_premises(
                emirates_id,
                history_source=history_source,
                live_audit_sources=live_audit_sources,
                row_writer=row_writer,
            )
            summary["status"] = "SUCCESS"
            summary["started_at"] = started
            summary["finished_at"] = datetime.now().isoformat(timespec="seconds")
            summaries.append(summary)
        except Exception as exc:
            print(f"ERROR processing {emirates_id}: {exc}")
            summaries.append({
                "emirates_id": emirates_id,
                "history_source": history_source,
                "live_audit_sources": list(live_audit_sources),
                "status": "ERROR",
                "error": str(exc),
                "started_at": started,
                "finished_at": datetime.now().isoformat(timespec="seconds"),
            })

        save_audit_summaries(summaries, paths["summary_path"])
        print(f"Updated audit files. Rows written so far: {total_rows}")

    final_row_count = finalize_audit_json(paths["jsonl_path"], paths["json_path"])
    print("\nDONE")
    print(f"Rows: {final_row_count}")
    print(f"CSV: {paths['csv_path']}")
    print(f"JSONL: {paths['jsonl_path']}")
    print(f"JSON: {paths['json_path']}")
    print(f"Summary: {paths['summary_path']}")

    try:
        import pandas as pd
        from IPython.display import display
        display(pd.read_csv(paths["csv_path"]).head(50))
    except Exception:
        pass

    return [], summaries, output_dir


## Run Audit

Choose the audit source:

- `latest-successes`: scan `runs/ejari_creation_*/progress.json` and use the newest file with `ejari_creation_success` records.
- `uploaded-success-file`: select one or more creation `progress.json` files in the file picker. Valid files are merged before audit; invalid files are skipped only after confirmation.
- `live`: select one or more live API sources by checkbox: contract history only active, contract history all other than pending, and/or owner-assets leased+rented.


In [ ]:
# @title Success progress source helpers
from notebook_progress_utils import (
    count_progress_success_records,
    find_latest_ejari_creation_success_progress,
    iter_progress_success_records,
    load_progress_file,
    resolve_uploaded_success_progress_paths,
)

SUCCESS_PROGRESS_LIST_TYPES_FOR_DETAIL = ("leased", "rented", "owned")


def infer_property_type_id_from_success(record):
    explicit = first_present(record.get("property_type_id"), record.get("PropertyTypeId"), record.get("PropertyType"))
    if explicit:
        try:
            return int(explicit)
        except Exception:
            return explicit
    title = str(record.get("title") or record.get("Title") or "").strip().lower()
    if title.startswith("building"):
        return 2
    if title.startswith("unit"):
        return 3
    return None


def get_property_detail_for_success(dld_token, property_row_value, property_type_id):
    if not property_row_value or not property_type_id:
        return None, None, None, "Missing property_row_value or property_type_id; property detail was not fetched."

    errors = []
    for list_type in SUCCESS_PROGRESS_LIST_TYPES_FOR_DETAIL:
        try:
            data, url, detail_list_type_used = get_property_detail(dld_token, list_type, property_type_id, property_row_value)
            return data, url, detail_list_type_used, ""
        except Exception as exc:
            errors.append(f"{list_type}: {exc}")
    return None, None, None, "Property detail failed for success record: " + " || ".join(errors)


def history_match_for_success(record, history_index):
    lookup_order = [
        ("contract_row_value", record.get("contract_row_value")),
        ("property_id", record.get("property_id")),
        ("property_row_value", record.get("property_row_value")),
        ("contract_number", record.get("contract_number")),
    ]
    for bucket, key in lookup_order:
        if key is None:
            continue
        matches = history_index.get(bucket, {}).get(str(key).strip()) or []
        if matches:
            return matches[0]
    return None


def audit_success_progress_dewa_premises(progress_path, history_source="actual", row_writer=None):
    progress_path = Path(progress_path)
    progress = load_progress_file(progress_path)
    success_records = list(iter_progress_success_records(progress, progress_path))
    by_owner = {}
    for item in success_records:
        by_owner.setdefault(item["emirates_id"], []).append(item)

    rows = [] if row_writer is None else None
    summaries = []
    total_rows = 0

    selected_emirates_ids = select_emirates_ids_for_section("DEWA success progress audit", sorted(by_owner), default_all=True)
    for emirates_id in selected_emirates_ids:
        items = by_owner[emirates_id]
        print(f"\nProcessing success records for owner Emirates ID: {emirates_id}")
        get_active_bearer_token(force_refresh=True)
        dld_token = get_dld_token(emirates_id)

        try:
            history_contracts, _, history_url = get_contract_history(dld_token, source=history_source)
            history_index = build_history_index(history_contracts)
        except Exception as exc:
            print(f"Contract history lookup failed for {emirates_id}: {exc}")
            history_url = ""
            history_index = {}

        contract_details_cache = {}
        premise_status_cache = {}
        owner_row_count = 0
        property_detail_success_count = 0
        contract_history_match_count = 0

        for item in items:
            record = item["record"]
            property_id = record.get("property_id")
            property_row_value = record.get("property_row_value") or item.get("property_key")
            property_type_id = infer_property_type_id_from_success(record)
            title = property_title_from_sources(record)
            contract_row_value = record.get("contract_row_value")
            contract_number = record.get("contract_number")

            property_detail_response, property_detail_url, property_detail_list_type_used, property_detail_error = get_property_detail_for_success(
                dld_token,
                property_row_value,
                property_type_id,
            )
            if property_detail_response is not None:
                property_detail_success_count += 1

            history_match = history_match_for_success(record, history_index)
            if history_match is not None:
                contract_history_match_count += 1

            contract_details, contract_details_url, contract_details_error = get_cached_contract_details(
                dld_token,
                contract_row_value,
                contract_details_cache,
            )
            resolved_contract_number = first_present(
                contract_number,
                contract_details.get("ContractNumber") if isinstance(contract_details, dict) else None,
                history_match.get("ContractNumber") if isinstance(history_match, dict) else None,
            )
            title = property_title_from_sources(record, property_detail_response, contract_details, history_match)
            contract_version_fields = contract_version_report_fields(contract_details, history_match, record, property_detail_response)

            dewa_premise_from_success_progress = join_values(extract_dewa_premises(record))
            dewa_premise_from_property_detail = join_values(extract_dewa_premises(property_detail_response))
            dewa_premise_from_contract_details = join_values(extract_dewa_premises(contract_details))
            dewa_premise_from_contract_history = join_values(extract_dewa_premises(history_match))
            premise_for_status_check = first_present(
                first_dewa_premise(property_detail_response),
                first_dewa_premise(contract_details),
                first_dewa_premise(record),
                first_dewa_premise(history_match),
            )
            premise_status_result = get_cached_premise_status(resolved_contract_number, premise_for_status_check, premise_status_cache)
            premise_status_fields = premise_status_report_fields(premise_status_result, premise_for_status_check)
            premise_mismatch = dewa_premise_mismatch_columns({
                "property_detail": dewa_premise_from_property_detail,
                "contract_details": dewa_premise_from_contract_details,
                "contract_history": dewa_premise_from_contract_history,
                "success_progress": dewa_premise_from_success_progress,
                "dewa_status_check": premise_status_fields.get("dewa_status_check_premise_no"),
            })

            presence = make_presence_columns(property_detail_response is not None, history_match is not None)
            presence["source_record_type"] = "progress_success"
            if property_detail_response is None:
                presence["source_mismatch"] = "PROGRESS_SUCCESS_PROPERTY_DETAIL_MISSING"

            row = {
                "emirates_id": emirates_id,
                "property_id": property_id,
                "property_row_value": property_row_value,
                "contract_number": resolved_contract_number,
                **contract_version_fields,
                **presence,
                "dewa_premise_from_property_detail": dewa_premise_from_property_detail,
                "dewa_premise_from_contract_details": dewa_premise_from_contract_details,
                "dewa_premise_from_contract_history": dewa_premise_from_contract_history,
                "dewa_premise_from_properties_list": "",
                "dewa_premise_from_success_progress": dewa_premise_from_success_progress,
                **premise_mismatch,
                **premise_status_fields,
                "properties_list_type": "",
                "property_type_id": property_type_id,
                "property_title": title,
                "contract_row_value": contract_row_value,
                "contract_status_from_properties_list": "",
                "contract_status_from_history": contract_status_text(history_match) if isinstance(history_match, dict) else None,
                "contract_history_role": history_match.get("_history_role") if isinstance(history_match, dict) else None,
                "property_detail_list_type_used": property_detail_list_type_used,
                "properties_list_url": "",
                "property_detail_url": property_detail_url,
                "contract_details_url": contract_details_url,
                "contract_history_url": history_url,
                "property_detail_error": property_detail_error,
                "contract_details_error": contract_details_error,
                "progress_success_file": str(progress_path),
                "progress_success_property_key": item.get("property_key"),
            }
            emit_audit_row(row, rows, row_writer)
            owner_row_count += 1
            total_rows += 1

        summaries.append({
            "emirates_id": emirates_id,
            "history_source": history_source,
            "source": "progress_success",
            "progress_success_file": str(progress_path),
            "success_records": owner_row_count,
            "property_detail_success_count": property_detail_success_count,
            "contract_history_match_count": contract_history_match_count,
            "output_rows": owner_row_count,
        })
        print(f"Success records: {owner_row_count} | property detail fetched: {property_detail_success_count} | active-history matches: {contract_history_match_count}")

    return rows or [], summaries


def run_dewa_premise_audit_from_success_progress(progress_path, history_source="actual"):
    output_dir = make_output_dir("dewa_premise_audit_successes")
    paths = initialize_audit_outputs(output_dir)
    summaries = []
    total_rows = 0

    print(f"Success progress file: {progress_path}")
    print(f"Run folder: {output_dir}")
    print(f"Live CSV: {paths['csv_path']}")
    print(f"Live JSONL: {paths['jsonl_path']}")

    def row_writer(row):
        nonlocal total_rows
        append_audit_row(row, paths["csv_path"], paths["jsonl_path"])
        total_rows += 1

    started = datetime.now().isoformat(timespec="seconds")
    try:
        _, summaries = audit_success_progress_dewa_premises(progress_path, history_source=history_source, row_writer=row_writer)
        for summary in summaries:
            summary["status"] = "SUCCESS"
            summary["started_at"] = started
            summary["finished_at"] = datetime.now().isoformat(timespec="seconds")
    except Exception as exc:
        print(f"ERROR processing success progress file: {exc}")
        summaries.append({
            "source": "progress_success",
            "progress_success_file": str(progress_path),
            "history_source": history_source,
            "status": "ERROR",
            "error": str(exc),
            "started_at": started,
            "finished_at": datetime.now().isoformat(timespec="seconds"),
        })

    save_audit_summaries(summaries, paths["summary_path"])
    final_row_count = finalize_audit_json(paths["jsonl_path"], paths["json_path"])
    print("\nDONE")
    print(f"Rows: {final_row_count}")
    print(f"CSV: {paths['csv_path']}")
    print(f"JSONL: {paths['jsonl_path']}")
    print(f"JSON: {paths['json_path']}")
    print(f"Summary: {paths['summary_path']}")

    try:
        import pandas as pd
        from IPython.display import display
        display(pd.read_csv(paths["csv_path"]).head(50))
    except Exception:
        pass

    return [], summaries, output_dir


In [ ]:
# @title Run DEWA premise audit
source_aliases = {
    "1": "latest-successes",
    "latest": "latest-successes",
    "latest-success": "latest-successes",
    "recent": "latest-successes",
    "recent-successes": "latest-successes",
    "2": "uploaded-success-file",
    "upload": "uploaded-success-file",
    "uploaded": "uploaded-success-file",
    "uploads": "uploaded-success-file",
    "multiple": "uploaded-success-file",
    "multiple-files": "uploaded-success-file",
    "file": "uploaded-success-file",
    "files": "uploaded-success-file",
    "success-file": "uploaded-success-file",
    "success-files": "uploaded-success-file",
    "3": "live",
    "current": "live",
    "existing": "live",
    "as-is": "live",
    "live": "live",
    "no": "no",
    "n": "no",
}


def choose_contract_history_source(prompt="Contract history API source?"):
    return choose_option(
        prompt,
        [
            ("actual", "Actual API"),
            ("proxy", "Proxy API"),
        ],
        default="actual",
        aliases={"1": "actual", "2": "proxy"},
        title="Contract history source",
    )


def choose_live_dewa_audit_sources():
    return choose_multiple_options(
        "Select live DEWA audit source(s). You can select more than one.",
        [
            (LIVE_DEWA_SOURCE_CONTRACT_HISTORY_ACTIVE, "Contract history (only active)"),
            (LIVE_DEWA_SOURCE_CONTRACT_HISTORY_NON_PENDING, "Contract history (all other than pending)"),
            (LIVE_DEWA_SOURCE_OWNER_ASSETS, "Owner assets (leased + rented)"),
        ],
        default=DEFAULT_LIVE_DEWA_AUDIT_SOURCES,
        aliases={
            "1": LIVE_DEWA_SOURCE_CONTRACT_HISTORY_ACTIVE,
            "active": LIVE_DEWA_SOURCE_CONTRACT_HISTORY_ACTIVE,
            "2": LIVE_DEWA_SOURCE_CONTRACT_HISTORY_NON_PENDING,
            "non-pending": LIVE_DEWA_SOURCE_CONTRACT_HISTORY_NON_PENDING,
            "not-pending": LIVE_DEWA_SOURCE_CONTRACT_HISTORY_NON_PENDING,
            "3": LIVE_DEWA_SOURCE_OWNER_ASSETS,
            "owner-assets": LIVE_DEWA_SOURCE_OWNER_ASSETS,
            "leased-rented": LIVE_DEWA_SOURCE_OWNER_ASSETS,
        },
        title="Live DEWA audit sources",
    )


audit_source = choose_option(
    "DEWA audit source?",
    [
        ("latest-successes", "Most recent progress.json successes"),
        ("uploaded-success-file", "Select progress.json file(s)"),
        ("live", "Live API audit"),
        ("no", "Skip audit"),
    ],
    default="live",
    aliases=source_aliases,
    title="DEWA audit source",
)

if audit_source == "latest-successes":
    progress_path, success_count = find_latest_ejari_creation_success_progress()
    print(f"Using latest Ejari creation success progress file: {progress_path} ({success_count} successes)")
    history_source = choose_contract_history_source("Contract history API source for optional active-history matching?")
    rows, summaries, output_dir = run_dewa_premise_audit_from_success_progress(progress_path, history_source=history_source)

elif audit_source == "uploaded-success-file":
    print("Select one or more Ejari creation progress.json files with top-level Emirates IDs and ejari_creation_success entries.")
    print("Do not select a single success_*.json detail file when you want a multi-property audit.")
    selected_progress_paths = select_file_paths(
        "Select one or more Ejari creation progress.json files",
        default_dir=".",
        multiple=True,
        title="Select progress.json file(s)",
    )
    progress_path, success_count, merge_dir, valid_files, invalid_files = resolve_uploaded_success_progress_paths(
        selected_progress_paths,
        confirm_callback=ask_yes_no,
        merge_output_prefix="dewa_uploaded_success_progress_merge",
    )
    if merge_dir:
        print(f"Using merged progress file: {progress_path} ({success_count} successes)")
    else:
        print(f"Using progress file: {progress_path} ({success_count} successes)")
    history_source = choose_contract_history_source("Contract history API source for optional active-history matching?")
    rows, summaries, output_dir = run_dewa_premise_audit_from_success_progress(progress_path, history_source=history_source)

elif audit_source == "live":
    live_audit_sources = choose_live_dewa_audit_sources()
    history_source = "actual"
    if any(source in live_audit_sources for source in (LIVE_DEWA_SOURCE_CONTRACT_HISTORY_ACTIVE, LIVE_DEWA_SOURCE_CONTRACT_HISTORY_NON_PENDING)):
        history_source = choose_contract_history_source("Contract history API source?")
    rows, summaries, output_dir = run_dewa_premise_audit(
        OWNER_EMIRATES_IDS,
        history_source=history_source,
        live_audit_sources=live_audit_sources,
    )

elif audit_source == "no":
    print("Skipped DEWA premise ID audit.")

else:
    raise ValueError("Invalid audit source. Choose latest-successes, uploaded-success-file, live, or no.")
